In [ ]:
from bs4 import BeautifulSoup
import requests
from urllib.parse import urljoin 
import os

folders = [
    "./data",
    "./data/raw_data",
    "./data/processed",
    "./images",
    "./images/1_star",
    "./images/2_star",
    "./images/3_star",
    "./images/4_star",
    "./images/5_star",
    "./raw_images"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

base_url = "https://books.toscrape.com/catalogue/page-1.html"
next_page_url = base_url
current_page = 1
raw_books = []

while next_page_url and current_page <= 3:
    response = requests.get(next_page_url)
    soup = BeautifulSoup(response.content, "html.parser")
    
    articles = soup.find_all("article", class_="product_pod")
    len(articles)

    for book in articles:
        name = book.h3.a["title"]
        price = book.find("p", class_="price_color").text[2:]
        rating = book.find("p", class_="star-rating")["class"][1]
        img_src = book.find("img")["src"]

        img_url = urljoin(base_url, img_src)
        img_content = requests.get(img_url).content

        image_name = name[:50].replace(':', '') + ".jpg"
        image_path = f"./raw_images/{image_name}"

        with open(image_path, "wb") as image_file:
            image_file.write(img_content)
        
        try:
            raw_books.append(
                {
                    "title": name, 
                    "price": price, 
                    "rating": rating,
                    "image_path": image_path
                }
            )
        except Exception as e:
            print(f"Error processing book '{name}': {e}")
        
    print(f"Page {current_page} - completed ----------------------------------")

    next_button = soup.find("li", class_="next")
    current_page += 1
    next_page_url = f"https://books.toscrape.com/catalogue/page-{current_page}.html" if next_button else None


print("Scrapping Done !")

Page 1 - completed ----------------------------------
Page 2 - completed ----------------------------------
Page 3 - completed ----------------------------------
Scrapping Done !


In [2]:
import pandas as pd
import os
import shutil

df = pd.DataFrame(raw_books)
df.to_csv("./data/raw_data/raw_books.csv", index=False)

rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["rating"] = df["rating"].map(rating_map)

df["price"] = df["price"].astype(float)

os.makedirs("./data/processed", exist_ok=True)

df.to_csv("./data/processed/processed_books.csv", index=False)

groups = df.groupby("rating")

for rating, group in groups:
    file_path = f"./data/processed/{rating}_star.csv"
    group.to_csv(file_path, index=False)

for _, row in df.iterrows():
    rating = row["rating"]
    src_path = row["image_path"]
    
    image_name = os.path.basename(src_path)
    
    dest_folder = f"./images/{rating}_star"
    dest_path = os.path.join(dest_folder, image_name)

    os.makedirs(dest_folder, exist_ok=True)

    try:
        shutil.move(src_path, dest_path)
    except Exception as e:
        print(f"Error moving {image_name}: {e}")

print("Processing Done !")

Processing Done !
